# Statistical Arbitrage: Pairs Trading with Cointegration

**Market-Neutral Statistical Arbitrage using ARBS Framework**

This notebook demonstrates **cointegration-based pairs trading** integrated with the ARBS backtesting framework.

## What You'll Learn

1. **ARBS Integration**: Implementing PairsSignal extending BaseSignal
2. **Cointegration Testing**: Engle-Granger methodology for pair selection
3. **Signal Generation**: Z-score based mean-reversion signals
4. **Portfolio Construction**: Using MeanVarianceOptimizer with pairs signals
5. **Full Pipeline**: BaseSignal → AlphaGenerator → Optimizer → MinimalBacktest → TearSheet
6. **Performance Analysis**: Market-neutral returns with comprehensive tear sheet

## Key Concepts

**Cointegration**: Two non-stationary price series that have a stationary linear combination (spread)

**Pairs Trading**: Long undervalued asset, short overvalued asset, profit when spread mean-reverts

**Market-Neutral**: Dollar-neutral positions eliminate market exposure (beta ≈ 0)

**Paper References**:
- Engle & Granger (1987): "Co-integration and Error Correction"
- Gatev, Goetzmann & Rouwenhorst (2006): "Pairs Trading: Performance of a Relative-Value Arbitrage Rule"
- Do & Faff (2010): "Does Simple Pairs Trading Still Work?"

---

## Setup

In [ ]:
# Add parent directory to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Standard imports
import numpy as np
import pandas as pd
import polars as pl
from datetime import date, timedelta
from typing import List, Tuple, Dict, Optional

# ARBS Framework Imports
from src.signals.base import BaseSignal
from src.alpha.generator import AlphaGenerator
from src.risk.ledoit_wolf import LedoitWolfShrinkage
from src.optimizer.mean_variance import MeanVarianceOptimizer
from src.backtest.minimal import MinimalBacktest
from src.accounting.tearsheet import TearSheet
from src.returns.calculator import ReturnsCalculator
from src.volatility.estimator import RealizedVolatility

# Statistical tests
from statsmodels.tsa.stattools import coint
from sklearn.linear_model import LinearRegression

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Set random seed for reproducibility
np.random.seed(42)

print("✓ ARBS Framework loaded successfully!")

---

## Part 1: Generate Mock Stock Data

Create synthetic cointegrated stock data for testing.

In [ ]:
# Define stocks and sectors
stocks = {
    'AAPL': 'Technology',
    'MSFT': 'Technology',
    'GOOGL': 'Technology',
    'NVDA': 'Technology',
    'JPM': 'Financials',
    'BAC': 'Financials',
    'GS': 'Financials',
    'XOM': 'Energy',
    'CVX': 'Energy',
    'COP': 'Energy'
}

tickers = list(stocks.keys())
n_days = 504
dates = [date(2022, 1, 1) + timedelta(days=i) for i in range(n_days)]

def generate_cointegrated_prices(n_days: int, stocks_dict: Dict[str, str]) -> Dict[str, np.ndarray]:
    """
    Generate price series with cointegration within sectors.
    
    Strategy:
    1. Generate sector-level common factor (random walk)
    2. Each stock = sector_factor + idiosyncratic_factor
    3. Within sector, stocks share cointegration relationship
    """
    # Group stocks by sector
    sector_stocks = {}
    for ticker, sector in stocks_dict.items():
        if sector not in sector_stocks:
            sector_stocks[sector] = []
        sector_stocks[sector].append(ticker)
    
    prices = {}
    
    # Generate prices for each sector
    for sector, sector_tickers in sector_stocks.items():
        n_stocks = len(sector_tickers)
        
        # Generate common sector factor (non-stationary random walk)
        sector_shocks = np.random.normal(0, 0.015, n_days)
        sector_factor = 100 + np.cumsum(sector_shocks)
        
        # Generate mean-reverting component (stationary)
        phi = 0.95  # Mean reversion speed
        mean_reverting = np.zeros(n_days)
        for t in range(1, n_days):
            shock = np.random.normal(0, 0.5)
            mean_reverting[t] = phi * mean_reverting[t-1] + shock
        
        # Each stock = sector_factor + stock_specific + mean_reverting
        for i, ticker in enumerate(sector_tickers):
            drift = np.random.normal(0, 0.001, n_days).cumsum()
            idiosyncratic = np.random.normal(0, 0.005, n_days).cumsum()
            
            stock_price = (
                0.85 * sector_factor +
                0.15 * idiosyncratic +
                0.3 * mean_reverting +
                drift
            )
            
            stock_price = np.maximum(stock_price, 10)
            prices[ticker] = stock_price
    
    return prices

# Generate cointegrated prices
prices_dict = generate_cointegrated_prices(n_days, stocks)

# Create Polars DataFrame
prices_data = []
for ticker in tickers:
    for i, d in enumerate(dates):
        prices_data.append({
            'date': d,
            'ticker': ticker,
            'price': prices_dict[ticker][i],
            'sector': stocks[ticker]
        })

prices_df = pl.DataFrame(prices_data)

print(f"✓ Generated {n_days} days of cointegrated price data for {len(tickers)} stocks")

---

## Part 2: Implement PairsSignal (ARBS Integration)

**Key ARBS Pattern**: Extend BaseSignal to implement custom signal logic.

PairsSignal workflow:
1. Find cointegrated pairs using Engle-Granger test
2. Calculate spreads and z-scores for each pair
3. Generate mean-reversion signals
4. Return signal matrix compatible with AlphaGenerator

In [ ]:
class PairsSignal(BaseSignal):
    """
    Cointegration-based pairs trading signal.
    
    Workflow:
    1. Test all possible pairs for cointegration (Engle-Granger)
    2. For cointegrated pairs, calculate spread = price_A - β × price_B
    3. Calculate z-score = (spread - rolling_mean) / rolling_std
    4. Generate signals: z > +2.0 → short spread, z < -2.0 → long spread
    5. Exit when z crosses 0
    
    Args:
        prices_df: Polars DataFrame with columns [date, ticker, price]
        min_correlation: Minimum correlation for pair consideration
        significance_level: P-value threshold for cointegration test
        z_score_window: Rolling window for z-score calculation
        entry_threshold: Z-score threshold for entry (default: 2.0)
        exit_threshold: Z-score threshold for exit (default: 0.0)
    """
    
    def __init__(self,
                 prices_df: pl.DataFrame,
                 min_correlation: float = 0.70,
                 significance_level: float = 0.05,
                 z_score_window: int = 20,
                 entry_threshold: float = 2.0,
                 exit_threshold: float = 0.0):
        self.prices_df = prices_df
        self.min_correlation = min_correlation
        self.significance_level = significance_level
        self.z_score_window = z_score_window
        self.entry_threshold = entry_threshold
        self.exit_threshold = exit_threshold
        
        # Store cointegrated pairs and their metadata
        self.pairs = []
        self.hedge_ratios = {}
        
    def _test_cointegration(self, price_A: np.ndarray, price_B: np.ndarray) -> Tuple[bool, float]:
        """Test if two price series are cointegrated."""
        t_stat, p_value, _ = coint(price_A, price_B)
        
        # Estimate hedge ratio via OLS
        model = LinearRegression()
        model.fit(price_B.reshape(-1, 1), price_A)
        hedge_ratio = model.coef_[0]
        
        cointegrated = p_value < self.significance_level
        
        return cointegrated, hedge_ratio
    
    def _find_cointegrated_pairs(self, prices_wide: pd.DataFrame) -> None:
        """Find all cointegrated pairs."""
        tickers = prices_wide.columns.tolist()
        
        # Calculate correlation matrix
        corr_matrix = prices_wide.corr()
        
        # Test all pairs with sufficient correlation
        for i, ticker_A in enumerate(tickers):
            for ticker_B in tickers[i+1:]:
                # Check correlation first (screening)
                corr = corr_matrix.loc[ticker_A, ticker_B]
                if corr < self.min_correlation:
                    continue
                
                # Test cointegration
                price_A = prices_wide[ticker_A].values
                price_B = prices_wide[ticker_B].values
                
                cointegrated, hedge_ratio = self._test_cointegration(price_A, price_B)
                
                if cointegrated:
                    pair_name = f"{ticker_A}-{ticker_B}"
                    self.pairs.append((ticker_A, ticker_B))
                    self.hedge_ratios[pair_name] = hedge_ratio
    
    def _calculate_z_score(self, spread: np.ndarray) -> np.ndarray:
        """Calculate rolling z-score for spread."""
        spread_series = pd.Series(spread)
        
        rolling_mean = spread_series.rolling(window=self.z_score_window, min_periods=self.z_score_window).mean()
        rolling_std = spread_series.rolling(window=self.z_score_window, min_periods=self.z_score_window).std()
        
        z_score = (spread_series - rolling_mean) / rolling_std
        
        return z_score.values
    
    def _generate_pair_signals(self, z_score: np.ndarray) -> np.ndarray:
        """Generate trading signals from z-scores."""
        positions = np.zeros(len(z_score))
        current_position = 0
        
        for i in range(len(z_score)):
            if np.isnan(z_score[i]):
                positions[i] = current_position
                continue
            
            # Entry signals
            if current_position == 0:
                if z_score[i] > self.entry_threshold:
                    current_position = -1  # Short spread
                elif z_score[i] < -self.entry_threshold:
                    current_position = 1   # Long spread
            
            # Exit signals
            elif current_position == 1:
                if z_score[i] > self.exit_threshold:
                    current_position = 0
            elif current_position == -1:
                if z_score[i] < self.exit_threshold:
                    current_position = 0
            
            positions[i] = current_position
        
        return positions
    
    def generate(self) -> pl.DataFrame:
        """
        Generate signals for all cointegrated pairs.
        
        Returns:
            Polars DataFrame with columns [date, ticker, signal]
            where ticker = "PAIR_A-B" and signal is the z-score based position
        """
        # Convert to wide format for cointegration testing
        prices_wide = self.prices_df.pivot(
            index='date',
            columns='ticker',
            values='price'
        ).to_pandas()
        
        dates_list = prices_wide.index.tolist()
        
        # Find cointegrated pairs
        self._find_cointegrated_pairs(prices_wide)
        
        if len(self.pairs) == 0:
            print("⚠️  No cointegrated pairs found. Using top correlated pairs as fallback.")
            # Fallback: use top 3 correlated pairs
            corr_matrix = prices_wide.corr()
            pairs_list = []
            tickers = prices_wide.columns.tolist()
            for i, ticker_A in enumerate(tickers):
                for ticker_B in tickers[i+1:]:
                    pairs_list.append((ticker_A, ticker_B, corr_matrix.loc[ticker_A, ticker_B]))
            pairs_list.sort(key=lambda x: x[2], reverse=True)
            
            for ticker_A, ticker_B, _ in pairs_list[:3]:
                price_A = prices_wide[ticker_A].values
                price_B = prices_wide[ticker_B].values
                _, hedge_ratio = self._test_cointegration(price_A, price_B)
                
                self.pairs.append((ticker_A, ticker_B))
                self.hedge_ratios[f"{ticker_A}-{ticker_B}"] = hedge_ratio
        
        # Generate signals for each pair
        signals_data = []
        
        for ticker_A, ticker_B in self.pairs:
            pair_name = f"{ticker_A}-{ticker_B}"
            hedge_ratio = self.hedge_ratios[pair_name]
            
            # Calculate spread
            price_A = prices_wide[ticker_A].values
            price_B = prices_wide[ticker_B].values
            spread = price_A - hedge_ratio * price_B
            
            # Calculate z-score
            z_score = self._calculate_z_score(spread)
            
            # Generate signals
            positions = self._generate_pair_signals(z_score)
            
            # Store signals (use z-score as raw signal, position as directional signal)
            for i, d in enumerate(dates_list):
                signals_data.append({
                    'date': d,
                    'ticker': pair_name,
                    'signal': positions[i]  # -1, 0, or +1
                })
        
        print(f"✓ Generated signals for {len(self.pairs)} cointegrated pairs")
        
        return pl.DataFrame(signals_data)

# Test PairsSignal
pairs_signal = PairsSignal(
    prices_df=prices_df,
    min_correlation=0.70,
    significance_level=0.05,
    z_score_window=20,
    entry_threshold=2.0,
    exit_threshold=0.0
)

signals_df = pairs_signal.generate()

print(f"\n✓ PairsSignal generated {len(signals_df)} signal observations")
print(f"Pairs found: {', '.join([f'{a}-{b}' for a, b in pairs_signal.pairs])}")

---

## Part 3: Calculate Returns for Pairs

Use ReturnsCalculator to prepare returns data for backtesting.

In [ ]:
# Create synthetic returns for pairs based on price data
# In practice, pair returns = position × (ret_A - β × ret_B)

# Convert prices to wide format
prices_wide = prices_df.pivot(
    index='date',
    columns='ticker',
    values='price'
).to_pandas()

# Calculate pair returns
pair_returns_data = []

for ticker_A, ticker_B in pairs_signal.pairs:
    pair_name = f"{ticker_A}-{ticker_B}"
    hedge_ratio = pairs_signal.hedge_ratios[pair_name]
    
    # Get returns
    ret_A = prices_wide[ticker_A].pct_change()
    ret_B = prices_wide[ticker_B].pct_change()
    
    # Pair return = ret_A - β × ret_B (spread return)
    pair_return = ret_A - hedge_ratio * ret_B
    
    for i, d in enumerate(prices_wide.index):
        pair_returns_data.append({
            'date': d,
            'ticker': pair_name,
            'return': pair_return.iloc[i]
        })

returns_df = pl.DataFrame(pair_returns_data).drop_nulls()

print(f"✓ Calculated returns for {len(pairs_signal.pairs)} pairs")
print(f"Returns shape: {len(returns_df)} observations")

---

## Part 4: Full ARBS Pipeline

**Pipeline**: PairsSignal → AlphaGenerator → LedoitWolfShrinkage → MeanVarianceOptimizer → MinimalBacktest → TearSheet

This demonstrates the full integration with ARBS framework.

In [ ]:
# Step 1: Alpha Generation (IC × Vol × Z)
alpha_generator = AlphaGenerator(
    IC=0.05,  # 5% IC (typical for pairs strategies)
    vol_estimator=RealizedVolatility(lookback=60, annualization_factor=252)
)

print("✓ AlphaGenerator configured (IC = 0.05)")

# Step 2: Covariance Estimation
cov_estimator = LedoitWolfShrinkage()

print("✓ LedoitWolfShrinkage covariance estimator configured")

# Step 3: Portfolio Optimization
optimizer = MeanVarianceOptimizer(
    risk_aversion=3.0,  # Moderate risk aversion
    long_only=False,    # Allow short positions
    leverage_limit=1.5  # Max 1.5x leverage
)

print("✓ MeanVarianceOptimizer configured (risk_aversion=3.0, long_only=False)")

# Step 4: Run Backtest
backtest = MinimalBacktest(
    signals_df=signals_df,
    returns_df=returns_df,
    alpha_generator=alpha_generator,
    cov_estimator=cov_estimator,
    optimizer=optimizer
)

print("\n🚀 Running backtest...")
result = backtest.run()

print(f"\n✓ Backtest complete!")
print(f"Total Return: {result.total_return:.2%}")
print(f"Sharpe Ratio: {result.sharpe_ratio:.3f}")
print(f"Information Coefficient: {result.IC:.4f}")

---

## Part 5: Performance Analysis with TearSheet

Generate comprehensive performance analysis using ARBS TearSheet.

In [ ]:
# Create TearSheet
tearsheet = TearSheet(
    returns=result.returns,
    signals_df=signals_df,
    returns_df=returns_df
)

# Generate full tear sheet
print("\n" + "="*80)
print("PAIRS TRADING STRATEGY - PERFORMANCE TEAR SHEET")
print("="*80)

tearsheet.plot_all()

print("\n💡 Key Insights:")
print("  • Pairs trading is market-neutral (beta ≈ 0)")
print("  • Returns driven by mean-reversion in cointegrated spreads")
print("  • ARBS framework enables modular testing of different pair selection methods")
print("  • AlphaGenerator scales signals appropriately using IC × Vol × Z formula")

---

## Part 6: Comparison with Market

Compare pairs strategy (market-neutral) vs long-only market portfolio.

In [ ]:
# Calculate market returns (equal-weight all stocks)
market_returns = prices_wide.pct_change().mean(axis=1).dropna()

# Align dates
pairs_returns = result.returns.to_pandas()
common_dates = pairs_returns.index.intersection(market_returns.index)

pairs_returns_aligned = pairs_returns.loc[common_dates]
market_returns_aligned = market_returns.loc[common_dates]

# Calculate cumulative returns
cum_pairs = (1 + pairs_returns_aligned).cumprod()
cum_market = (1 + market_returns_aligned).cumprod()

# Calculate beta
cov_matrix = np.cov(pairs_returns_aligned, market_returns_aligned)
beta = cov_matrix[0, 1] / cov_matrix[1, 1]

# Plot comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Panel 1: Cumulative returns
axes[0].plot(cum_pairs.index, cum_pairs.values, linewidth=2.5, color='green',
            label='Pairs Strategy (Market-Neutral)', alpha=0.9)
axes[0].plot(cum_market.index, cum_market.values, linewidth=2.5, color='steelblue',
            label='Market (Long-Only)', alpha=0.7)
axes[0].axhline(1, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Pairs Trading vs Market - Cumulative Returns', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Cumulative Return', fontsize=10)
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Panel 2: Scatter plot (correlation)
axes[1].scatter(market_returns_aligned, pairs_returns_aligned, alpha=0.5, s=20)
axes[1].axhline(0, color='black', linestyle='-', linewidth=1)
axes[1].axvline(0, color='black', linestyle='-', linewidth=1)
axes[1].set_xlabel('Market Return', fontsize=10)
axes[1].set_ylabel('Pairs Strategy Return', fontsize=10)
axes[1].set_title(f'Market Neutrality (β = {beta:.3f})', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print metrics
market_sharpe = market_returns_aligned.mean() / market_returns_aligned.std() * np.sqrt(252)

print(f"\n📊 Strategy Comparison:")
print(f"\nPairs Strategy:")
print(f"  Total Return: {result.total_return:.2%}")
print(f"  Sharpe Ratio: {result.sharpe_ratio:.3f}")
print(f"  Beta (vs Market): {beta:.3f}")

market_total_return = (1 + market_returns_aligned).prod() - 1
print(f"\nMarket Portfolio:")
print(f"  Total Return: {market_total_return:.2%}")
print(f"  Sharpe Ratio: {market_sharpe:.3f}")
print(f"  Beta (vs Market): 1.000")

print(f"\n💡 Pairs strategy beta = {beta:.3f} confirms market-neutral positioning")

---

## Summary: ARBS Integration

### What We Demonstrated

**✓ Custom Signal Implementation**:
- Created `PairsSignal` extending `BaseSignal`
- Implemented cointegration testing (Engle-Granger)
- Generated z-score based mean-reversion signals

**✓ Full Pipeline Integration**:
- `PairsSignal` → `AlphaGenerator` (IC × Vol × Z scaling)
- `LedoitWolfShrinkage` → `MeanVarianceOptimizer` (portfolio construction)
- `MinimalBacktest` → `TearSheet` (performance analysis)

**✓ Market-Neutral Strategy**:
- Beta ≈ 0 (uncorrelated with market)
- Returns from mean-reversion, not market direction
- Lower volatility than long-only strategies

### Key ARBS Patterns Used

1. **BaseSignal Extension**: Custom signal logic while maintaining framework compatibility
2. **Modular Components**: Each stage (signals, alpha, risk, optimizer) can be swapped independently
3. **Standardized Data Flow**: Polars DataFrames with consistent schemas throughout pipeline
4. **Comprehensive Analysis**: TearSheet provides production-ready performance metrics

### Production Considerations

- **Transaction Costs**: Add proportional costs in optimizer
- **Cointegration Monitoring**: Re-test pairs periodically for relationship breakdown
- **Dynamic Hedge Ratios**: Update β using rolling window or Kalman filter
- **Risk Limits**: Add max position constraints per pair

**Paper References**:
- Engle & Granger (1987): "Co-integration and Error Correction"
- Gatev et al. (2006): "Pairs Trading: Performance of a Relative-Value Arbitrage Rule"
- Grinold & Kahn (1999): "Active Portfolio Management" (Alpha generation framework)